# Dev check: in-schedule VoltageOffset flux pulse vs. SetParameter idle bias

**Not a numbered calibration node.** This notebook is Phase 0 of the calibration-node
rollout plan: it checks whether a schedule-internal `VoltageOffset` flux pulse can
coexist with the existing external `SetParameter` idle bias, and whether the voltage
correctly returns to that idle bias once the pulse ends.

**Before running on real hardware:** start with a small `FLUX_PULSE_START`/`FLUX_PULSE_STOP`
range (well inside a range you already trust from `run05_resonator_flux_spectroscopy`), and
only widen it after confirming the sweet-spot curve below matches a SetParameter-based
`ResonatorFluxSpectroscopy` run of the same qubit.

In [1]:
from pathlib import Path

from qblox_lab.config.hardware import create_hardware_agent
from qblox_lab.config.sessions import CONFIG_DIR, SESSIONS
from qblox_lab.experiments.dev_joint_flux_pulse_check import JointFluxPulseCheck

## Run parameters

In [4]:
SESSION = SESSIONS["AS_QRC"]
HARDWARE_CONFIG = SESSION.hardware_config
DEVICE_CONFIG = SESSION.device_config
FLUX_CONFIG = SESSION.flux_config # Idle bias applied before the pulse sweep
OUTPUT_DIR = Path("data")

QUBIT = "q1"
FREQUENCY_CENTER = None
FREQUENCY_WIDTH = 30e6
FREQUENCY_POINTS = 201

# Conservative first pass: a small delta on top of the already-applied idle bias.
# Widen only after this range reproduces the expected sweet-spot shift.
FLUX_PULSE_START = -0.05
FLUX_PULSE_STOP = 0.05
FLUX_PULSE_POINTS = 11
REPETITIONS = 5
PULSE_SETTLE_TIME = 1e-6
IDLE_RETURN_TIME = 1e-6

TIMEOUT = 300
CREATE_DUMMY_CONNECTIONS = False

PLOT_RESULTS = True

## Hardware and experiment

In [5]:
hardware_agent = create_hardware_agent(
    hardware_configuration=HARDWARE_CONFIG,
    device_configuration=DEVICE_CONFIG,
    output_dir=OUTPUT_DIR,
    create_dummy_connections=CREATE_DUMMY_CONNECTIONS,
)

experiment = JointFluxPulseCheck(
    hardware_agent=hardware_agent,
    qubit=QUBIT,
    flux_config=FLUX_CONFIG,
)

## Measurement

In [6]:
dataset = experiment.run_measurement(
    frequency_center=FREQUENCY_CENTER,
    frequency_width=FREQUENCY_WIDTH,
    frequency_points=FREQUENCY_POINTS,
    flux_pulse_start=FLUX_PULSE_START,
    flux_pulse_stop=FLUX_PULSE_STOP,
    flux_pulse_points=FLUX_PULSE_POINTS,
    repetitions=REPETITIONS,
    pulse_settle_time=PULSE_SETTLE_TIME,
    idle_return_time=IDLE_RETURN_TIME,
    timeout=TIMEOUT,
)
dataset

KeyError: 'q1:fl was not found in the connectivity.'

## Analysis

In [ ]:
results = experiment.analysis()
for qubit, result in results.items():
    if result.success:
        print(f"{qubit}: period={result.flux_period:.6g}, "
              f"center={result.center_frequency / 1e9:.9f} GHz")
        for index, value in enumerate(result.sweet_spots):
            print(f"  candidate {index}: {value:.6f} V (relative to idle bias)")
    else:
        print(f"{qubit}: fit failed")

In [ ]:
if PLOT_RESULTS:
    experiment.plot()

## Next step

Compare the fitted period/sweet-spot candidates above against a
`run05_resonator_flux_spectroscopy` run of the same qubit and an equivalent range. If they
agree, the VoltageOffset mechanism is validated for building the coupler-flux and future
CZ-chevron nodes in the rollout plan.